In [ ]:
from google.colab import drive
from pathlib import Path

# Connect Google Drive
drive.mount("/content/drive")

# Location of the CICIDS2017 CSV files
data_folder = Path(
    "/content/drive/MyDrive/IDPS_Research/"
    "CICIDS2017/MachineLearningCSV"
)

# Find all CSV files
csv_files = sorted(data_folder.glob("*.csv"))

print("Folder exists:", data_folder.exists())
print("Number of CSV files:", len(csv_files))

for file in csv_files:
    print(file.name)


In [ ]:
import pandas as pd

print("CICIDS2017 — Initial Dataset Inspection")
print("=" * 65)

total_rows = 0

for file in csv_files:
    # Read only the column names
    columns = pd.read_csv(file, nrows=0).columns

    # Count data rows without loading the full CSV
    with open(file, "rb") as f:
        row_count = sum(1 for _ in f) - 1

    total_rows += row_count

    print(f"\nFile: {file.name}")
    print(f"Rows: {row_count:,}")
    print(f"Columns: {len(columns)}")

print("\n" + "=" * 65)
print(f"TOTAL RECORDS: {total_rows:,}")
print(f"TOTAL CSV FILES: {len(csv_files)}")


In [ ]:
import pandas as pd
from collections import Counter

print("CICIDS2017 — Attack Label Inspection")
print("=" * 60)

all_labels = Counter()

for file in csv_files:
    # Read only the label column
    header = pd.read_csv(file, nrows=0).columns

    label_column = next(
        col for col in header
        if col.strip().lower() == "label"
    )

    labels = pd.read_csv(
        file,
        usecols=[label_column],
        dtype="string"
    )[label_column].str.strip()

    counts = labels.value_counts(dropna=False)

    print(f"\n{file.name}")
    print(counts.to_string())

    all_labels.update(counts.to_dict())

print("\n" + "=" * 60)
print("OVERALL CLASS DISTRIBUTION")

for label, count in all_labels.most_common():
    print(f"{str(label):25s} {count:>10,}")

print("\nTotal records:", sum(all_labels.values()))


In [ ]:
import pandas as pd
import numpy as np

print("CICIDS2017 — Preliminary Data Quality Audit")
print("=" * 65)

for file in csv_files:
    df = pd.read_csv(file, nrows=10000, low_memory=False)

    # Remove extra spaces from column names
    df.columns = df.columns.str.strip()

    # Identify numeric columns
    numeric_df = df.select_dtypes(include="number")

    missing = int(df.isna().sum().sum())
    infinite = int(
        np.isinf(numeric_df.to_numpy()).sum()
    )

    print(f"\nFile: {file.name}")
    print(f"Sample rows: {len(df):,}")
    print(f"Missing values: {missing:,}")
    print(f"Infinite values: {infinite:,}")

print("\nPreliminary audit completed.")


In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

missing_by_feature = Counter()
infinite_by_feature = Counter()

print("CICIDS2017 — Problematic Feature Inspection")
print("=" * 60)

for file in csv_files:
    df = pd.read_csv(file, nrows=10000, low_memory=False)
    df.columns = df.columns.str.strip()

    numeric_df = df.select_dtypes(include="number")

    for col, count in df.isna().sum().items():
        if count > 0:
            missing_by_feature[col] += int(count)

    for col in numeric_df.columns:
        count = np.isinf(numeric_df[col].to_numpy()).sum()
        if count > 0:
            infinite_by_feature[col] += int(count)

print("\nFEATURES WITH MISSING VALUES")
for feature, count in missing_by_feature.most_common():
    print(f"{feature}: {count:,}")

print("\nFEATURES WITH INFINITE VALUES")
for feature, count in infinite_by_feature.most_common():
    print(f"{feature}: {count:,}")

print("\nInspection completed.")


In [ ]:
import pandas as pd
import numpy as np

print("CICIDS2017 — Missing-Value Consistency Check")
print("=" * 70)

grand_total = 0

for file in csv_files:
    df = pd.read_csv(file, nrows=10000, low_memory=False)
    df.columns = df.columns.str.strip()

    # Two equivalent ways to count missing cells
    method_1 = int(df.isna().sum().sum())
    method_2 = int(sum(df[col].isna().sum() for col in df.columns))

    grand_total += method_1

    print(f"\n{file.name}")
    print("Method 1:", method_1)
    print("Method 2:", method_2)
    print("Match:", method_1 == method_2)

    affected = df.isna().sum()
    affected = affected[affected > 0]

    if not affected.empty:
        print("Affected columns:")
        print(affected.to_string())

print("\n" + "=" * 70)
print("TOTAL MISSING CELLS:", grand_total)


In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

missing_counts = Counter()
infinite_counts = Counter()
total_rows_checked = 0

print("CICIDS2017 — Full Dataset Quality Audit")
print("=" * 65)

for file in csv_files:
    file_rows = 0

    for chunk in pd.read_csv(
        file,
        chunksize=50000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()

        file_rows += len(chunk)

        # Count missing values in all columns
        for col, count in chunk.isna().sum().items():
            if count:
                missing_counts[col] += int(count)

        # Detect numeric infinities, including in columns
        # that may have been read as text
        for col in chunk.columns:
            if col == "Label":
                continue

            numeric = pd.to_numeric(
                chunk[col], errors="coerce"
            )

            count = int(
                np.isinf(numeric.to_numpy()).sum()
            )

            if count:
                infinite_counts[col] += count

    total_rows_checked += file_rows

    print(f"{file.name}: {file_rows:,} rows checked")

print("\n" + "=" * 65)
print(f"TOTAL ROWS CHECKED: {total_rows_checked:,}")

print("\nMISSING VALUES BY FEATURE")
for feature, count in missing_counts.most_common():
    print(f"{feature}: {count:,}")

print("\nINFINITE VALUES BY FEATURE")
for feature, count in infinite_counts.most_common():
    print(f"{feature}: {count:,}")

print("\nTOTAL MISSING CELLS:", sum(missing_counts.values()))
print("TOTAL INFINITE CELLS:", sum(infinite_counts.values()))


In [ ]:
import pandas as pd
import numpy as np

samples = []

for file in csv_files:
    df = pd.read_csv(file, nrows=10000, low_memory=False)
    df.columns = df.columns.str.strip()

    # Keep the source filename for reporting
    # but exclude it from duplicate comparisons
    df["source_file"] = file.name
    samples.append(df)

audit_df = pd.concat(samples, ignore_index=True)

# Standardize numeric infinities for comparison
audit_df = audit_df.replace([np.inf, -np.inf], np.nan)

feature_cols = [
    col for col in audit_df.columns
    if col not in ["Label", "source_file"]
]

exact_duplicates = audit_df.duplicated(
    subset=feature_cols + ["Label"]
).sum()

feature_duplicates = audit_df.duplicated(
    subset=feature_cols
).sum()

# Count distinct labels for each feature pattern
label_counts = audit_df.groupby(
    feature_cols,
    dropna=False
)["Label"].nunique()

conflicting_groups = (label_counts > 1).sum()

print("CICIDS2017 — Preliminary Duplicate Audit")
print("=" * 60)
print("Sample records:", len(audit_df))
print("Exact duplicate rows:", int(exact_duplicates))
print("Repeated feature vectors:", int(feature_duplicates))
print("Conflicting-label feature groups:", int(conflicting_groups))


In [ ]:
import pandas as pd
import numpy as np

print("CICIDS2017 — Full Duplicate Audit")
print("=" * 60)

seen_features = set()
seen_full_rows = set()

total_records = 0
exact_duplicates = 0
feature_duplicates = 0

for file in csv_files:
    file_records = 0

    for chunk in pd.read_csv(
        file,
        chunksize=25000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()
        chunk = chunk.replace([np.inf, -np.inf], np.nan)

        feature_cols = [
            c for c in chunk.columns if c != "Label"
        ]

        feature_hashes = pd.util.hash_pandas_object(
            chunk[feature_cols], index=False
        ).to_numpy()

        full_hashes = pd.util.hash_pandas_object(
            chunk, index=False
        ).to_numpy()

        for fh, rh in zip(feature_hashes, full_hashes):
            fh = int(fh)
            rh = int(rh)

            if fh in seen_features:
                feature_duplicates += 1
            else:
                seen_features.add(fh)

            if rh in seen_full_rows:
                exact_duplicates += 1
            else:
                seen_full_rows.add(rh)

        file_records += len(chunk)

    total_records += file_records
    print(f"Checked: {file.name} ({file_records:,} rows)")

print("\n" + "=" * 60)
print(f"Total records: {total_records:,}")
print(f"Exact duplicate rows: {exact_duplicates:,}")
print(f"Repeated feature vectors: {feature_duplicates:,}")
print(f"Unique feature patterns: {len(seen_features):,}")


In [ ]:
import pandas as pd
import numpy as np

# For each feature hash, track its observed labels
labels_by_feature = {}

print("CICIDS2017 — Conflicting Label Audit")
print("=" * 65)

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=25000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()
        chunk = chunk.replace([np.inf, -np.inf], np.nan)

        feature_cols = [
            c for c in chunk.columns if c != "Label"
        ]

        hashes = pd.util.hash_pandas_object(
            chunk[feature_cols], index=False
        ).to_numpy()

        labels = chunk["Label"].astype(str).str.strip()

        for feature_hash, label in zip(hashes, labels):
            key = int(feature_hash)

            if key not in labels_by_feature:
                labels_by_feature[key] = set()

            labels_by_feature[key].add(label)

    print("Checked:", file.name)

multiclass_conflicts = 0
binary_conflicts = 0

for label_set in labels_by_feature.values():
    if len(label_set) > 1:
        multiclass_conflicts += 1

    binary_set = {
        0 if label.upper() == "BENIGN" else 1
        for label in label_set
    }

    if len(binary_set) > 1:
        binary_conflicts += 1

print("\n" + "=" * 65)
print("Unique feature patterns:", len(labels_by_feature))
print("Multiclass conflicting-label groups:", multiclass_conflicts)
print("Binary conflicting-label groups:", binary_conflicts)


In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

# Identify feature hashes with conflicting binary labels
conflicting_hashes = set()

for feature_hash, labels in labels_by_feature.items():
    binary_labels = {
        0 if str(label).upper() == "BENIGN" else 1
        for label in labels
    }

    if len(binary_labels) > 1:
        conflicting_hashes.add(feature_hash)

# Count records belonging to conflicting groups
conflicting_records = 0
benign_records = 0
attack_records = 0

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=25000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()
        chunk = chunk.replace([np.inf, -np.inf], np.nan)

        feature_cols = [
            c for c in chunk.columns if c != "Label"
        ]

        hashes = pd.util.hash_pandas_object(
            chunk[feature_cols], index=False
        ).to_numpy()

        labels = chunk["Label"].astype(str).str.strip()

        for feature_hash, label in zip(hashes, labels):
            if int(feature_hash) in conflicting_hashes:
                conflicting_records += 1

                if label.upper() == "BENIGN":
                    benign_records += 1
                else:
                    attack_records += 1

print("CICIDS2017 — Conflicting Group Size Audit")
print("=" * 60)
print("Conflicting feature groups:", len(conflicting_hashes))
print("Records in conflicting groups:", conflicting_records)
print("Benign records:", benign_records)
print("Attack records:", attack_records)
print(
    "Percentage of full dataset:",
    round(100 * conflicting_records / 2830743, 4),
    "%"
)


In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict

print("CICIDS2017 — Theoretical Error Analysis")
print("=" * 60)

# Reuse the conflicting hashes from Experiment 11
group_counts = defaultdict(lambda: [0, 0])

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=25000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()
        chunk = chunk.replace([np.inf, -np.inf], np.nan)

        feature_cols = [
            col for col in chunk.columns
            if col != "Label"
        ]

        hashes = pd.util.hash_pandas_object(
            chunk[feature_cols], index=False
        ).to_numpy()

        labels = chunk["Label"].astype(str).str.strip()

        for feature_hash, label in zip(hashes, labels):
            key = int(feature_hash)

            if key in conflicting_hashes:
                binary_label = (
                    0 if label.upper() == "BENIGN" else 1
                )
                group_counts[key][binary_label] += 1

# Minimum errors for each group = minority-label count
minimum_errors = sum(
    min(benign, attack)
    for benign, attack in group_counts.values()
)

total_conflicting_records = sum(
    benign + attack
    for benign, attack in group_counts.values()
)

print("Conflicting groups:", len(group_counts))
print("Records in conflicting groups:", total_conflicting_records)
print("Minimum unavoidable errors:", minimum_errors)

print(
    "Minimum error rate within conflicting groups:",
    round(100 * minimum_errors /
          total_conflicting_records, 4),
    "%"
)

print(
    "Minimum error rate across full dataset:",
    round(100 * minimum_errors /
          2830743, 4),
    "%"
)


In [ ]:
import pandas as pd
import numpy as np

print("CICIDS2017 — Experiment 13")
print("Proposed Duplicate-Aware Split")
print("=" * 60)

split_counts = {
    "Train": [0, 0],
    "Validation": [0, 0],
    "Test": [0, 0]
}

# Fixed constant makes the assignment reproducible
split_seed = np.uint64(42)

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=25000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()
        chunk = chunk.replace(
            [np.inf, -np.inf], np.nan
        )

        feature_cols = [
            c for c in chunk.columns
            if c != "Label"
        ]

        hashes = pd.util.hash_pandas_object(
            chunk[feature_cols],
            index=False
        ).to_numpy(dtype=np.uint64)

        # Deterministic assignment by feature hash
        buckets = (hashes ^ split_seed) % 10000

        labels = (
            chunk["Label"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        attack = (labels != "BENIGN").to_numpy()

        masks = {
            "Train": buckets < 7000,
            "Validation": (
                (buckets >= 7000) &
                (buckets < 8500)
            ),
            "Test": buckets >= 8500
        }

        for name, mask in masks.items():
            split_counts[name][0] += int(
                np.sum(mask & ~attack)
            )
            split_counts[name][1] += int(
                np.sum(mask & attack)
            )

    print("Processed:", file.name)

print("\n" + "=" * 60)
print("PROPOSED SPLIT DISTRIBUTION")

for name, (benign, attacks) in split_counts.items():
    total = benign + attacks
    print(f"\n{name}")
    print(f"  Total: {total:,}")
    print(f"  Benign: {benign:,}")
    print(f"  Attacks: {attacks:,}")
    print(f"  Attack percentage: {100*attacks/total:.2f}%")

print("\nGrand total:",
      sum(sum(v) for v in split_counts.values()))


In [ ]:
from collections import Counter
import pandas as pd
import numpy as np

category_counts = {
    "Train": Counter(),
    "Validation": Counter(),
    "Test": Counter()
}

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=25000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()
        chunk = chunk.replace(
            [np.inf, -np.inf], np.nan
        )

        feature_cols = [
            c for c in chunk.columns if c != "Label"
        ]

        hashes = pd.util.hash_pandas_object(
            chunk[feature_cols],
            index=False
        ).to_numpy(dtype=np.uint64)

        buckets = (hashes ^ np.uint64(42)) % 10000

        labels = chunk["Label"].astype(str).str.strip()

        for label, bucket in zip(labels, buckets):
            if bucket < 7000:
                partition = "Train"
            elif bucket < 8500:
                partition = "Validation"
            else:
                partition = "Test"

            category_counts[partition][label] += 1

summary = pd.DataFrame(category_counts).fillna(0).astype(int)
summary["Total"] = summary.sum(axis=1)

print("CICIDS2017 — Attack Categories by Partition")
print("=" * 65)
print(summary.to_string())

print("\nCategories absent from training:")
print(summary.index[summary["Train"] == 0].tolist())


In [ ]:
import pandas as pd
import numpy as np

print("CICIDS2017 — Experiment 15")
print("Cross-Partition Feature Overlap Audit")
print("=" * 65)

hash_to_partition = {}
overlap_conflicts = 0
records_checked = 0

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=25000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()
        chunk = chunk.replace(
            [np.inf, -np.inf], np.nan
        )

        feature_cols = [
            c for c in chunk.columns
            if c != "Label"
        ]

        hashes = pd.util.hash_pandas_object(
            chunk[feature_cols],
            index=False
        ).to_numpy(dtype=np.uint64)

        buckets = (hashes ^ np.uint64(42)) % 10000

        partitions = np.where(
            buckets < 7000, 0,
            np.where(buckets < 8500, 1, 2)
        )

        for h, partition in zip(hashes, partitions):
            key = int(h)
            partition = int(partition)

            if key in hash_to_partition:
                if hash_to_partition[key] != partition:
                    overlap_conflicts += 1
            else:
                hash_to_partition[key] = partition

        records_checked += len(chunk)

    print("Checked:", file.name)

print("\n" + "=" * 65)
print("Total records checked:", f"{records_checked:,}")
print("Unique feature hashes:", f"{len(hash_to_partition):,}")
print("Cross-partition hash conflicts:", overlap_conflicts)


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

output_folder = data_folder.parent / "experiment16"
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "provisional_split_assignments.csv"

first_chunk = True
saved_rows = 0

for file in csv_files:
    row_offset = 0

    for chunk in pd.read_csv(
        file,
        chunksize=25000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()
        chunk = chunk.replace(
            [np.inf, -np.inf], np.nan
        )

        feature_cols = [
            c for c in chunk.columns
            if c != "Label"
        ]

        hashes = pd.util.hash_pandas_object(
            chunk[feature_cols],
            index=False
        ).to_numpy(dtype=np.uint64)

        buckets = (hashes ^ np.uint64(42)) % 10000

        partitions = np.where(
            buckets < 7000, "Train",
            np.where(buckets < 8500, "Validation", "Test")
        )

        result = pd.DataFrame({
            "source_file": file.name,
            "row_number": np.arange(
                row_offset, row_offset + len(chunk)
            ),
            "feature_hash": [str(int(h)) for h in hashes],
            "partition": partitions
        })

        result.to_csv(
            output_file,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False
        )

        first_chunk = False
        row_offset += len(chunk)
        saved_rows += len(chunk)

    print("Saved assignments for:", file.name)

print("\nExperiment 16 completed.")
print("Rows saved:", f"{saved_rows:,}")
print("File:", output_file)


In [ ]:
import pandas as pd

split_file = (
    data_folder.parent
    / "experiment16"
    / "provisional_split_assignments.csv"
)

split_df = pd.read_csv(
    split_file,
    dtype={
        "source_file": "string",
        "row_number": "int64",
        "feature_hash": "string",
        "partition": "string"
    }
)

print("CICIDS2017 — Experiment 17")
print("Saved Split Verification")
print("=" * 60)

print("Total saved records:", f"{len(split_df):,}")

print("\nPartition counts:")
print(split_df["partition"].value_counts().to_string())

print("\nMissing values:")
print(split_df.isna().sum().to_string())

print("\nDuplicate source-file/row-number pairs:")
print(
    split_df.duplicated(
        subset=["source_file", "row_number"]
    ).sum()
)

print("\nInvalid partition names:")
valid_names = {"Train", "Validation", "Test"}
invalid = ~split_df["partition"].isin(valid_names)
print(int(invalid.sum()))

print("\nFeature hashes assigned to multiple partitions:")
hash_conflicts = (
    split_df.groupby("feature_hash")["partition"]
    .nunique()
    .gt(1)
    .sum()
)
print(int(hash_conflicts))

print("\nVerification complete.")


In [ ]:
import pandas as pd
from collections import defaultdict

print("CICIDS2017 — Experiment 18")
print("Feature Type Consistency Check")
print("=" * 65)

column_types = defaultdict(set)
column_orders = []

for file in csv_files:
    header = pd.read_csv(file, nrows=0).columns.str.strip()
    column_orders.append(tuple(header))

    # Compare types inferred from two different regions
    for skip in [0, 10000]:
        sample = pd.read_csv(
            file,
            skiprows=range(1, skip + 1),
            nrows=5000,
            low_memory=False
        )
        sample.columns = sample.columns.str.strip()

        for col, dtype in sample.dtypes.items():
            column_types[col].add(str(dtype))

    print("Checked:", file.name)

inconsistent = {
    col: sorted(types)
    for col, types in column_types.items()
    if len(types) > 1
}

print("\nAll files have identical column order:",
      len(set(column_orders)) == 1)

print("Columns with differing inferred types:",
      len(inconsistent))

for col, types in inconsistent.items():
    print(f"{col}: {types}")

print("\nInspection complete.")


In [ ]:
import pandas as pd
import numpy as np

print("CICIDS2017 — Experiment 19")
print("Full Feature Representation Consistency Audit")
print("=" * 65)

# Read the saved assignments
saved = pd.read_csv(
    data_folder.parent
    / "experiment16"
    / "provisional_split_assignments.csv",
    dtype={"feature_hash": "string"}
)

saved = saved.set_index(
    ["source_file", "row_number"]
)

hash_mismatches = 0
records_checked = 0
conversion_failures = 0

for file in csv_files:
    row_offset = 0

    for chunk in pd.read_csv(
        file,
        chunksize=25000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()

        feature_cols = [
            c for c in chunk.columns if c != "Label"
        ]

        # Fixed numeric interpretation of all features
        numeric = chunk[feature_cols].apply(
            pd.to_numeric, errors="coerce"
        )

        # Count non-missing values that failed conversion
        conversion_failures += int(
            (numeric.isna() & chunk[feature_cols].notna())
            .sum().sum()
        )

        numeric = numeric.replace(
            [np.inf, -np.inf], np.nan
        )

        new_hashes = pd.util.hash_pandas_object(
            numeric, index=False
        ).astype("uint64").astype(str).to_numpy()

        row_numbers = np.arange(
            row_offset, row_offset + len(chunk)
        )

        keys = pd.MultiIndex.from_arrays(
            [
                [file.name] * len(chunk),
                row_numbers
            ],
            names=["source_file", "row_number"]
        )

        old_hashes = (
            saved.loc[keys, "feature_hash"]
            .astype(str)
            .to_numpy()
        )

        hash_mismatches += int(
            np.sum(new_hashes != old_hashes)
        )

        records_checked += len(chunk)
        row_offset += len(chunk)

    print("Checked:", file.name)

print("\n" + "=" * 65)
print("Records checked:", f"{records_checked:,}")
print("Hash mismatches:", f"{hash_mismatches:,}")
print("Numeric conversion failures:",
      f"{conversion_failures:,}")


In [ ]:
import json
import shutil
from datetime import datetime, timezone

experiment20 = data_folder.parent / "experiment20"
experiment20.mkdir(parents=True, exist_ok=True)

source = (
    data_folder.parent
    / "experiment16"
    / "provisional_split_assignments.csv"
)

destination = experiment20 / "frozen_split_assignments.csv"

shutil.copy2(source, destination)

metadata = {
    "dataset": "CICIDS2017",
    "task": "Binary intrusion detection",
    "total_records": 2830743,
    "partition_method": "Deterministic feature-hash grouping",
    "hash_seed": 42,
    "target_split": {
        "Train": 0.70,
        "Validation": 0.15,
        "Test": 0.15
    },
    "observed_split_counts": {
        "Train": 1988583,
        "Validation": 420601,
        "Test": 421559
    },
    "feature_hash_overlap_conflicts": 0,
    "numeric_conversion_failures": 0,
    "hash_representation_mismatches": 0,
    "split_status": "Frozen for model development",
    "limitations": [
        "Hash collisions are theoretically possible",
        "This is not a temporal split",
        "This is not an unseen-attack evaluation",
        "Class proportions are not exactly stratified"
    ],
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

metadata_file = experiment20 / "split_metadata.json"

with open(metadata_file, "w") as f:
    json.dump(metadata, f, indent=2)

print("CICIDS2017 — Experiment 20")
print("=" * 60)
print("Frozen split saved:", destination.exists())
print("Metadata saved:", metadata_file.exists())
print("Split records:", f"{len(split_df):,}")
print("Experiment 20 completed.")


In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

print("CICIDS2017 — Experiment 21")
print("Training Partition Inspection")
print("=" * 65)

# Use the frozen assignments from Experiment 20
frozen_file = (
    data_folder.parent
    / "experiment20"
    / "frozen_split_assignments.csv"
)

assignments = pd.read_csv(
    frozen_file,
    usecols=["source_file", "row_number", "partition"]
)

train_counts = Counter()
train_rows = 0
feature_names = None

for file in csv_files:
    file_assignments = assignments.loc[
        assignments["source_file"] == file.name,
        ["row_number", "partition"]
    ].sort_values("row_number")

    partitions = file_assignments["partition"].to_numpy()
    offset = 0

    for chunk in pd.read_csv(
        file,
        chunksize=25000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()

        chunk_partitions = partitions[
            offset:offset + len(chunk)
        ]

        train_mask = chunk_partitions == "Train"
        train_chunk = chunk.loc[train_mask]

        if feature_names is None:
            feature_names = [
                c for c in chunk.columns if c != "Label"
            ]

        labels = (
            train_chunk["Label"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        train_counts["Benign"] += int(
            (labels == "BENIGN").sum()
        )

        train_counts["Attack"] += int(
            (labels != "BENIGN").sum()
        )

        train_rows += len(train_chunk)
        offset += len(chunk)

    print("Checked:", file.name)

print("\n" + "=" * 65)
print("Training records:", f"{train_rows:,}")
print("Benign records:", f"{train_counts['Benign']:,}")
print("Attack records:", f"{train_counts['Attack']:,}")
print("Input features:", len(feature_names))
print("Target column: Label")


In [ ]:
import pandas as pd
from pathlib import Path

print("CICIDS2017 — Experiment 22")
print("Export Training Partition")
print("=" * 60)

output_folder = data_folder.parent / "experiment22"
output_folder.mkdir(parents=True, exist_ok=True)

train_file = output_folder / "train_raw.csv"

first_write = True
saved_rows = 0

for file in csv_files:
    file_assignments = assignments.loc[
        assignments["source_file"] == file.name,
        ["row_number", "partition"]
    ].sort_values("row_number")

    partitions = file_assignments["partition"].to_numpy()
    offset = 0

    for chunk in pd.read_csv(
        file,
        chunksize=25000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()

        chunk_partitions = partitions[
            offset:offset + len(chunk)
        ]

        train_chunk = chunk.loc[
            chunk_partitions == "Train"
        ]

        if len(train_chunk) > 0:
            train_chunk.to_csv(
                train_file,
                mode="w" if first_write else "a",
                header=first_write,
                index=False
            )

            first_write = False
            saved_rows += len(train_chunk)

        offset += len(chunk)

    print("Processed:", file.name)

print("\n" + "=" * 60)
print("Training records saved:", f"{saved_rows:,}")
print("Training file exists:", train_file.exists())
print("Output file:", train_file)


In [ ]:
import pandas as pd

print("CICIDS2017 — Experiment 23")
print("Training File Integrity Check")
print("=" * 60)

train_file = (
    data_folder.parent
    / "experiment22"
    / "train_raw.csv"
)

# Read the header and a small sample
sample = pd.read_csv(train_file, nrows=5)
sample.columns = sample.columns.str.strip()

print("Training file exists:", train_file.exists())
print("Number of columns:", len(sample.columns))
print("Input features:", len(sample.columns) - 1)
print("Target column present:", "Label" in sample.columns)

# Count all records and labels in manageable chunks
total_rows = 0
benign_rows = 0
attack_rows = 0

for chunk in pd.read_csv(
    train_file,
    usecols=["Label"],
    chunksize=50000
):
    labels = chunk["Label"].astype(str).str.strip()

    total_rows += len(labels)
    benign_rows += int((labels == "BENIGN").sum())
    attack_rows += int((labels != "BENIGN").sum())

print("\nTraining records:", f"{total_rows:,}")
print("Benign records:", f"{benign_rows:,}")
print("Attack records:", f"{attack_rows:,}")

print("\nExpected training records: 1,988,583")
print("Row count matches:", total_rows == 1988583)

print("\nExperiment 23 completed.")


In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

print("CICIDS2017 — Experiment 24")
print("Training-Only Missing and Infinite Values")
print("=" * 60)

missing_counts = Counter()
infinite_counts = Counter()
rows_checked = 0

for chunk in pd.read_csv(
    train_file,
    chunksize=25000,
    low_memory=False
):
    chunk.columns = chunk.columns.str.strip()

    features = chunk.drop(columns=["Label"])

    for col, count in features.isna().sum().items():
        if count:
            missing_counts[col] += int(count)

    for col in features.columns:
        values = pd.to_numeric(
            features[col], errors="coerce"
        ).to_numpy()

        count = int(np.isinf(values).sum())

        if count:
            infinite_counts[col] += count

    rows_checked += len(chunk)

print("Training rows checked:", f"{rows_checked:,}")

print("\nMissing values:")
for col, count in missing_counts.most_common():
    print(f"{col}: {count:,}")

print("\nInfinite values:")
for col, count in infinite_counts.most_common():
    print(f"{col}: {count:,}")

print("\nTotal missing cells:",
      sum(missing_counts.values()))
print("Total infinite cells:",
      sum(infinite_counts.values()))

print("\nExperiment 24 completed.")


In [ ]:
import pandas as pd
import numpy as np
import json

print("CICIDS2017 — Experiment 25")
print("Training-Only Median Imputation Parameters")
print("=" * 65)

affected_features = [
    "Flow Bytes/s",
    "Flow Packets/s"
]

# Read only the two affected columns
train_rates = pd.read_csv(
    train_file,
    usecols=lambda c: c.strip() in affected_features,
    low_memory=False
)

train_rates.columns = train_rates.columns.str.strip()

# Convert infinities to missing values
train_rates = train_rates.replace(
    [np.inf, -np.inf], np.nan
)

# Calculate medians using training records only
training_medians = (
    train_rates.median(numeric_only=True)
    .to_dict()
)

output_folder = data_folder.parent / "experiment25"
output_folder.mkdir(parents=True, exist_ok=True)

median_file = output_folder / "training_medians.json"

with open(median_file, "w") as f:
    json.dump(training_medians, f, indent=2)

print("Training-derived medians:")

for feature, value in training_medians.items():
    print(f"{feature}: {value:.6f}")

print("\nMedian parameters saved:", median_file.exists())
print("File:", median_file)
print("\nExperiment 25 completed.")


In [ ]:
import pandas as pd
import numpy as np
import json

print("CICIDS2017 — Experiment 26")
print("Training Sample Imputation Test")
print("=" * 60)

# Load previously saved training-only medians
median_file = (
    data_folder.parent
    / "experiment25"
    / "training_medians.json"
)

with open(median_file, "r") as f:
    medians = json.load(f)

# Read a small training sample
sample = pd.read_csv(
    train_file,
    nrows=10000,
    low_memory=False
)

sample.columns = sample.columns.str.strip()

# Count problematic values before cleaning
before_missing = int(sample.isna().sum().sum())

numeric_before = sample.select_dtypes(include="number")
before_infinite = int(
    np.isinf(numeric_before.to_numpy()).sum()
)

# Work on a copy; original data remains unchanged
clean_sample = sample.copy()

clean_sample = clean_sample.replace(
    [np.inf, -np.inf], np.nan
)

for feature, median in medians.items():
    clean_sample[feature] = (
        clean_sample[feature].fillna(median)
    )

# Verify the result
after_missing = int(clean_sample.isna().sum().sum())

numeric_after = clean_sample.select_dtypes(include="number")
after_infinite = int(
    np.isinf(numeric_after.to_numpy()).sum()
)

print("Sample records:", len(sample))
print("Missing values before:", before_missing)
print("Infinite values before:", before_infinite)
print("Missing values after:", after_missing)
print("Infinite values after:", after_infinite)

print("\nOriginal sample unchanged:",
      sample is not clean_sample)

print("\nExperiment 26 completed.")


In [ ]:
import pandas as pd
import numpy as np
import json

print("CICIDS2017 — Experiment 27")
print("Full Training Dataset Imputation")
print("=" * 65)

# Load training-only medians
median_file = (
    data_folder.parent
    / "experiment25"
    / "training_medians.json"
)

with open(median_file, "r") as f:
    medians = json.load(f)

output_folder = data_folder.parent / "experiment27"
output_folder.mkdir(parents=True, exist_ok=True)

clean_train_file = output_folder / "train_clean.csv"

first_write = True
total_rows = 0
remaining_missing = 0
remaining_infinite = 0

for chunk in pd.read_csv(
    train_file,
    chunksize=25000,
    low_memory=False
):
    chunk.columns = chunk.columns.str.strip()

    # Convert infinities to missing values
    chunk = chunk.replace(
        [np.inf, -np.inf], np.nan
    )

    # Apply medians calculated from training data
    for feature, median in medians.items():
        chunk[feature] = chunk[feature].fillna(median)

    # Check for remaining problematic values
    remaining_missing += int(
        chunk.isna().sum().sum()
    )

    numeric = chunk.select_dtypes(include="number")

    remaining_infinite += int(
        np.isinf(numeric.to_numpy()).sum()
    )

    # Save cleaned records
    chunk.to_csv(
        clean_train_file,
        mode="w" if first_write else "a",
        header=first_write,
        index=False
    )

    first_write = False
    total_rows += len(chunk)

print("Training records processed:", f"{total_rows:,}")
print("Remaining missing values:", remaining_missing)
print("Remaining infinite values:", remaining_infinite)
print("Cleaned file saved:", clean_train_file.exists())
print("Output:", clean_train_file)
print("\nExperiment 27 completed.")


In [ ]:
import pandas as pd
import numpy as np

print("CICIDS2017 — Experiment 28")
print("Saved Clean Training File Verification")
print("=" * 60)

clean_train_file = (
    data_folder.parent
    / "experiment27"
    / "train_clean.csv"
)

total_rows = 0
total_missing = 0
total_infinite = 0
benign_count = 0
attack_count = 0
column_count = None

for chunk in pd.read_csv(
    clean_train_file,
    chunksize=25000,
    low_memory=False
):
    chunk.columns = chunk.columns.str.strip()

    if column_count is None:
        column_count = len(chunk.columns)

    total_rows += len(chunk)
    total_missing += int(chunk.isna().sum().sum())

    numeric = chunk.select_dtypes(include="number")
    total_infinite += int(
        np.isinf(numeric.to_numpy()).sum()
    )

    labels = chunk["Label"].astype(str).str.strip()

    benign_count += int((labels == "BENIGN").sum())
    attack_count += int((labels != "BENIGN").sum())

print("Records:", f"{total_rows:,}")
print("Columns:", column_count)
print("Benign:", f"{benign_count:,}")
print("Attacks:", f"{attack_count:,}")
print("Missing values:", total_missing)
print("Infinite values:", total_infinite)

print("\nIntegrity checks passed:",
      total_rows == 1988583
      and column_count == 79
      and benign_count == 1589180
      and attack_count == 399403
      and total_missing == 0
      and total_infinite == 0)

print("\nExperiment 28 completed.")


In [ ]:
import pandas as pd
import numpy as np
import json

print("CICIDS2017 — Experiment 29")
print("Validation Partition Export and Imputation")
print("=" * 65)

# Load medians fitted on training data only
median_file = (
    data_folder.parent
    / "experiment25"
    / "training_medians.json"
)

with open(median_file, "r") as f:
    medians = json.load(f)

output_folder = data_folder.parent / "experiment29"
output_folder.mkdir(parents=True, exist_ok=True)

validation_file = output_folder / "validation_clean.csv"

first_write = True
saved_rows = 0
missing_after = 0
infinite_after = 0
benign_count = 0
attack_count = 0

for file in csv_files:
    file_assignments = assignments.loc[
        assignments["source_file"] == file.name,
        ["row_number", "partition"]
    ].sort_values("row_number")

    partitions = file_assignments["partition"].to_numpy()
    offset = 0

    for chunk in pd.read_csv(
        file,
        chunksize=25000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()

        chunk_partitions = partitions[
            offset:offset + len(chunk)
        ]

        validation_chunk = chunk.loc[
            chunk_partitions == "Validation"
        ].copy()

        offset += len(chunk)

        if validation_chunk.empty:
            continue

        # Convert infinite values to missing
        validation_chunk.replace(
            [np.inf, -np.inf], np.nan,
            inplace=True
        )

        # Apply TRAINING-derived medians
        for feature, median in medians.items():
            validation_chunk[feature] = (
                validation_chunk[feature].fillna(median)
            )

        missing_after += int(
            validation_chunk.isna().sum().sum()
        )

        numeric = validation_chunk.select_dtypes(
            include="number"
        )

        infinite_after += int(
            np.isinf(numeric.to_numpy()).sum()
        )

        labels = (
            validation_chunk["Label"]
            .astype(str)
            .str.strip()
        )

        benign_count += int(
            (labels == "BENIGN").sum()
        )

        attack_count += int(
            (labels != "BENIGN").sum()
        )

        validation_chunk.to_csv(
            validation_file,
            mode="w" if first_write else "a",
            header=first_write,
            index=False
        )

        first_write = False
        saved_rows += len(validation_chunk)

    print("Processed:", file.name)

print("\n" + "=" * 65)
print("Validation records saved:", f"{saved_rows:,}")
print("Benign records:", f"{benign_count:,}")
print("Attack records:", f"{attack_count:,}")
print("Remaining missing values:", missing_after)
print("Remaining infinite values:", infinite_after)
print("Validation file saved:", validation_file.exists())
print("Output:", validation_file)

print("\nExperiment 29 completed.")


In [ ]:
import pandas as pd
import numpy as np

print("CICIDS2017 — Experiment 30")
print("Saved Validation File Integrity Check")
print("=" * 60)

validation_file = (
    data_folder.parent
    / "experiment29"
    / "validation_clean.csv"
)

total_rows = 0
benign_count = 0
attack_count = 0
missing_count = 0
infinite_count = 0
column_count = None

for chunk in pd.read_csv(
    validation_file,
    chunksize=25000,
    low_memory=False
):
    chunk.columns = chunk.columns.str.strip()

    if column_count is None:
        column_count = len(chunk.columns)

    total_rows += len(chunk)
    missing_count += int(chunk.isna().sum().sum())

    numeric = chunk.select_dtypes(include="number")
    infinite_count += int(
        np.isinf(numeric.to_numpy()).sum()
    )

    labels = chunk["Label"].astype(str).str.strip()

    benign_count += int((labels == "BENIGN").sum())
    attack_count += int((labels != "BENIGN").sum())

print("Validation records:", f"{total_rows:,}")
print("Columns:", column_count)
print("Benign records:", f"{benign_count:,}")
print("Attack records:", f"{attack_count:,}")
print("Missing values:", missing_count)
print("Infinite values:", infinite_count)

checks_passed = (
    total_rows == 420601
    and column_count == 79
    and benign_count == 340428
    and attack_count == 80173
    and missing_count == 0
    and infinite_count == 0
)

print("\nIntegrity checks passed:", checks_passed)
print("\nExperiment 30 completed.")


In [ ]:
import os
import psutil

print("CICIDS2017 — Experiment 31")
print("Colab Memory and CPU Inspection")
print("=" * 60)

memory = psutil.virtual_memory()

print("Total RAM:",
      round(memory.total / (1024**3), 2), "GB")

print("Available RAM:",
      round(memory.available / (1024**3), 2), "GB")

print("RAM currently used:",
      round(memory.used / (1024**3), 2), "GB")

print("CPU cores available:",
      os.cpu_count())

print("\nExperiment 31 completed.")


In [ ]:
import pandas as pd
import numpy as np
import psutil

print("CICIDS2017 — Experiment 32")
print("Memory-Efficient Pilot Training Data")
print("=" * 60)

clean_train_file = (
    data_folder.parent
    / "experiment27"
    / "train_clean.csv"
)

# Load a manageable pilot subset
pilot_df = pd.read_csv(
    clean_train_file,
    nrows=200000,
    low_memory=False
)

pilot_df.columns = pilot_df.columns.str.strip()

X_pilot = (
    pilot_df.drop(columns=["Label"])
    .to_numpy(dtype=np.float32)
)

y_pilot = (
    pilot_df["Label"]
    .astype(str)
    .str.strip()
    .ne("BENIGN")
    .astype(np.int8)
    .to_numpy()
)

print("Pilot records:", len(X_pilot))
print("Features:", X_pilot.shape[1])
print("Benign:", int((y_pilot == 0).sum()))
print("Attacks:", int((y_pilot == 1).sum()))

print(
    "Feature matrix memory:",
    round(X_pilot.nbytes / (1024**2), 2),
    "MB"
)

print(
    "Available RAM:",
    round(
        psutil.virtual_memory().available / (1024**3),
        2
    ),
    "GB"
)

print("\nExperiment 32 completed.")


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time
import psutil

print("CICIDS2017 — Experiment 33")
print("Pilot Random Forest Training")
print("=" * 60)

pilot_rf = RandomForestClassifier(
    n_estimators=30,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=2
)

start_time = time.perf_counter()

pilot_rf.fit(X_pilot, y_pilot)

training_time = time.perf_counter() - start_time

print("Pilot training completed:", True)
print("Trees trained:", len(pilot_rf.estimators_))
print("Training time (seconds):",
      round(training_time, 2))

print(
    "Available RAM after training:",
    round(
        psutil.virtual_memory().available / (1024**3),
        2
    ),
    "GB"
)

print("\nExperiment 33 completed.")


In [ ]:
import pandas as pd
import numpy as np
import psutil
import gc

print("CICIDS2017 — Experiment 34")
print("Full Training Data — Memory Check")
print("=" * 60)

# Release pilot data and model to recover memory
del pilot_rf, pilot_df, X_pilot, y_pilot
gc.collect()

clean_train_file = (
    data_folder.parent
    / "experiment27"
    / "train_clean.csv"
)

# Read the complete cleaned training dataset
train_df = pd.read_csv(
    clean_train_file,
    low_memory=False
)

train_df.columns = train_df.columns.str.strip()

feature_names = [
    c for c in train_df.columns
    if c != "Label"
]

X_train = train_df[
    feature_names
].to_numpy(dtype=np.float32)

y_train = (
    train_df["Label"]
    .astype(str)
    .str.strip()
    .ne("BENIGN")
    .astype(np.int8)
    .to_numpy()
)

# Release the temporary DataFrame
del train_df
gc.collect()

print("Training records:", len(X_train))
print("Input features:", X_train.shape[1])

print("Benign:", int((y_train == 0).sum()))
print("Attacks:", int((y_train == 1).sum()))

print(
    "Feature matrix memory:",
    round(X_train.nbytes / (1024**2), 2),
    "MB"
)

print(
    "Available RAM:",
    round(
        psutil.virtual_memory().available / (1024**3),
        2
    ),
    "GB"
)

print("\nExperiment 34 completed.")


In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time
import psutil

print("CICIDS2017 — Experiment 35")
print("Full Random Forest Baseline Training")
print("=" * 65)

rf_baseline = RandomForestClassifier(
    n_estimators=30,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=2
)

start = time.perf_counter()

rf_baseline.fit(X_train, y_train)

rf_training_seconds = time.perf_counter() - start

print("Training completed:", True)
print("Training records:", f"{len(X_train):,}")
print("Features:", X_train.shape[1])
print("Trees trained:", len(rf_baseline.estimators_))
print("Training time (seconds):",
      round(rf_training_seconds, 2))

print(
    "Available RAM after training:",
    round(
        psutil.virtual_memory().available / (1024**3),
        2
    ),
    "GB"
)

print("\nExperiment 35 completed.")


In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("CICIDS2017 — Experiment 36")
print("Random Forest Baseline — Validation Evaluation")
print("=" * 65)

# Load the cleaned validation partition
validation_file = (
    data_folder.parent
    / "experiment29"
    / "validation_clean.csv"
)

val_df = pd.read_csv(
    validation_file,
    low_memory=False
)

val_df.columns = val_df.columns.str.strip()

# Use exactly the same feature order as training
X_val = val_df[
    feature_names
].to_numpy(dtype=np.float32)

y_val = (
    val_df["Label"]
    .astype(str)
    .str.strip()
    .ne("BENIGN")
    .astype(np.int8)
    .to_numpy()
)

# Measure prediction time
start = time.perf_counter()
y_pred = rf_baseline.predict(X_val)
prediction_seconds = time.perf_counter() - start

# Confusion matrix:
# 0 = benign, 1 = attack
tn, fp, fn, tp = confusion_matrix(
    y_val, y_pred, labels=[0, 1]
).ravel()

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, zero_division=0)
recall = recall_score(y_val, y_pred, zero_division=0)
macro_f1 = f1_score(
    y_val, y_pred,
    average="macro",
    zero_division=0
)
fpr = fp / (fp + tn)

print("Validation records:", f"{len(y_val):,}")
print("Accuracy:", round(accuracy, 4))
print("Attack precision:", round(precision, 4))
print("Attack recall:", round(recall, 4))
print("Macro-F1:", round(macro_f1, 4))
print("False-positive rate:", round(fpr, 4))
print("Missed attacks:", f"{fn:,}")
print("Prediction time (seconds):",
      round(prediction_seconds, 2))

print("\nConfusion matrix:")
print(f"TN={tn:,}  FP={fp:,}")
print(f"FN={fn:,}  TP={tp:,}")

print("\nExperiment 36 completed.")


In [ ]:
import json
import pandas as pd
from pathlib import Path

print("CICIDS2017 — Experiment 37")
print("Save Random Forest Baseline Evidence")
print("=" * 60)

output_folder = data_folder.parent / "experiment37"
output_folder.mkdir(parents=True, exist_ok=True)

baseline_results = {
    "dataset": "CICIDS2017",
    "evaluation_partition": "Validation",
    "training_records": int(len(y_train)),
    "validation_records": int(len(y_val)),
    "features": int(X_train.shape[1]),
    "model": "RandomForestClassifier",
    "n_estimators": 30,
    "max_depth": 15,
    "min_samples_leaf": 2,
    "random_state": 42,
    "n_jobs": 2,
    "accuracy": float(accuracy),
    "attack_precision": float(precision),
    "attack_recall": float(recall),
    "macro_f1": float(macro_f1),
    "false_positive_rate": float(fpr),
    "true_negatives": int(tn),
    "false_positives": int(fp),
    "false_negatives": int(fn),
    "true_positives": int(tp),
    "training_seconds": float(rf_training_seconds),
    "prediction_seconds": float(prediction_seconds)
}

json_file = output_folder / "rf_baseline_validation.json"
csv_file = output_folder / "rf_baseline_validation.csv"

with open(json_file, "w") as f:
    json.dump(baseline_results, f, indent=2)

pd.DataFrame([baseline_results]).to_csv(
    csv_file, index=False
)

print("JSON saved:", json_file.exists())
print("CSV saved:", csv_file.exists())
print("Baseline macro-F1:", round(macro_f1, 4))
print("Baseline FPR:", round(fpr, 4))
print("Baseline missed attacks:", fn)
print("\nExperiment 37 completed.")


In [ ]:
import numpy as np

print("CICIDS2017 — Experiment 38")
print("Stratified Training Sample for MI")
print("=" * 60)

rng = np.random.default_rng(42)

# Select records separately from each training class
benign_indices = np.flatnonzero(y_train == 0)
attack_indices = np.flatnonzero(y_train == 1)

sample_size = 100000

attack_n = round(
    sample_size * len(attack_indices) / len(y_train)
)
benign_n = sample_size - attack_n

selected_benign = rng.choice(
    benign_indices, size=benign_n, replace=False
)
selected_attack = rng.choice(
    attack_indices, size=attack_n, replace=False
)

selected_indices = np.concatenate(
    [selected_benign, selected_attack]
)
rng.shuffle(selected_indices)

X_mi_pilot = X_train[selected_indices]
y_mi_pilot = y_train[selected_indices]

print("Sample records:", len(y_mi_pilot))
print("Input features:", X_mi_pilot.shape[1])
print("Benign records:", int((y_mi_pilot == 0).sum()))
print("Attack records:", int((y_mi_pilot == 1).sum()))

print(
    "Sample attack percentage:",
    round(100 * y_mi_pilot.mean(), 2), "%"
)

print(
    "Full training attack percentage:",
    round(100 * y_train.mean(), 2), "%"
)

print("\nExperiment 38 completed.")


In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.feature_selection import mutual_info_classif

print("CICIDS2017 — Experiment 39")
print("Pilot Mutual Information Feature Ranking")
print("=" * 65)

start = time.perf_counter()

mi_scores = mutual_info_classif(
    X_mi_pilot,
    y_mi_pilot,
    discrete_features=False,
    n_neighbors=3,
    random_state=42
)

mi_seconds = time.perf_counter() - start

mi_ranking = pd.DataFrame({
    "Feature": feature_names,
    "MI_Score": mi_scores
}).sort_values(
    "MI_Score", ascending=False
).reset_index(drop=True)

mi_ranking.index += 1

print("Features ranked:", len(mi_ranking))
print("MI calculation time (seconds):",
      round(mi_seconds, 2))

print("\nTop 20 features:")
print(mi_ranking.head(20).to_string())

print("\nFeatures with zero MI score:",
      int((mi_ranking["MI_Score"] == 0).sum()))

print("\nExperiment 39 completed.")


In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.feature_selection import mutual_info_classif

print("CICIDS2017 — Experiment 40")
print("MI Feature Ranking Stability")
print("=" * 65)

# Reuse the Experiment 39 ranking for seed 42
rankings = {42: mi_ranking["Feature"].tolist()}
times = {42: mi_seconds}

for seed in [43, 44]:
    rng = np.random.default_rng(seed)

    selected_benign = rng.choice(
        benign_indices,
        size=benign_n,
        replace=False
    )

    selected_attack = rng.choice(
        attack_indices,
        size=attack_n,
        replace=False
    )

    selected = np.concatenate(
        [selected_benign, selected_attack]
    )
    rng.shuffle(selected)

    X_sample = X_train[selected]
    y_sample = y_train[selected]

    start = time.perf_counter()

    scores = mutual_info_classif(
        X_sample,
        y_sample,
        discrete_features=False,
        n_neighbors=3,
        random_state=seed
    )

    elapsed = time.perf_counter() - start

    order = np.argsort(-scores)
    rankings[seed] = [
        feature_names[i] for i in order
    ]
    times[seed] = elapsed

    print(
        f"Seed {seed}: completed in {elapsed:.2f} seconds"
    )

top30_sets = [
    set(rankings[seed][:30])
    for seed in [42, 43, 44]
]

intersection = set.intersection(*top30_sets)
union = set.union(*top30_sets)

stability = len(intersection) / len(union)

print("\nTop-30 stability results:")
print("Features common to all 3 rankings:",
      len(intersection))
print("Features in union:", len(union))
print("Intersection/union stability:",
      round(stability, 4))

print("\nMI calculation times:")
for seed in [42, 43, 44]:
    print(f"Seed {seed}: {times[seed]:.2f} seconds")

print("\nExperiment 40 completed.")


In [ ]:
import json
import pandas as pd

print("CICIDS2017 — Experiment 41")
print("Save Stable MI-30 Feature Set")
print("=" * 60)

# Use the Experiment 39 ranking (seed 42).
# Experiment 40 confirmed that its top-30 set
# also appeared in the other two rankings.
selected_features_30 = rankings[42][:30]

output_folder = data_folder.parent / "experiment41"
output_folder.mkdir(parents=True, exist_ok=True)

json_file = output_folder / "mi_top30_features.json"
csv_file = output_folder / "mi_top30_features.csv"

with open(json_file, "w") as f:
    json.dump(
        {
            "dataset": "CICIDS2017",
            "selection_method": "Mutual Information",
            "selection_data": "Training only",
            "ranking_seed": 42,
            "number_of_features": 30,
            "features": selected_features_30,
            "top30_common_across_seeds": [42, 43, 44],
            "intersection_union_stability": 1.0
        },
        f,
        indent=2
    )

pd.DataFrame({
    "Rank": range(1, 31),
    "Feature": selected_features_30
}).to_csv(csv_file, index=False)

print("Selected features:", len(selected_features_30))
print("Unique feature names:",
      len(set(selected_features_30)))
print("All features present in training:",
      all(f in feature_names for f in selected_features_30))
print("JSON saved:", json_file.exists())
print("CSV saved:", csv_file.exists())

print("\nSelected top-30 features:")
for i, feature in enumerate(selected_features_30, 1):
    print(f"{i:2d}. {feature}")

print("\nExperiment 41 completed.")


In [ ]:
import numpy as np
import time
import psutil
from sklearn.ensemble import RandomForestClassifier

print("CICIDS2017 — Experiment 42")
print("MI-30 Random Forest Training")
print("=" * 65)

# Find the selected feature positions
selected_indices_30 = [
    feature_names.index(f)
    for f in selected_features_30
]

# Select exactly the 30 saved MI features
X_train_30 = np.ascontiguousarray(
    X_train[:, selected_indices_30],
    dtype=np.float32
)

print("Training records:", f"{len(X_train_30):,}")
print("Selected features:", X_train_30.shape[1])

rf_mi30 = RandomForestClassifier(
    n_estimators=30,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=2
)

start = time.perf_counter()

rf_mi30.fit(X_train_30, y_train)

mi30_training_seconds = time.perf_counter() - start

print("\nTraining completed:", True)
print("Trees trained:", len(rf_mi30.estimators_))
print("Training time (seconds):",
      round(mi30_training_seconds, 2))

print(
    "Available RAM after training:",
    round(
        psutil.virtual_memory().available / (1024**3),
        2
    ),
    "GB"
)

print("\nExperiment 42 completed.")


In [ ]:

import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("CICIDS2017 — Experiment 43")
print("MI-30 vs All-78 Validation Comparison")
print("=" * 65)

# Use the validation data already loaded in Experiment 36
# and exactly the selected feature order from Experiment 41.
X_val_30 = np.ascontiguousarray(
    X_val[:, selected_indices_30],
    dtype=np.float32
)

start = time.perf_counter()
y_pred_30 = rf_mi30.predict(X_val_30)
mi30_prediction_seconds = time.perf_counter() - start

tn30, fp30, fn30, tp30 = confusion_matrix(
    y_val, y_pred_30, labels=[0, 1]
).ravel()

mi30_accuracy = accuracy_score(y_val, y_pred_30)
mi30_precision = precision_score(
    y_val, y_pred_30, zero_division=0
)
mi30_recall = recall_score(
    y_val, y_pred_30, zero_division=0
)
mi30_macro_f1 = f1_score(
    y_val, y_pred_30,
    average="macro",
    zero_division=0
)
mi30_fpr = fp30 / (fp30 + tn30)

comparison = pd.DataFrame({
    "Metric": [
        "Features",
        "Accuracy",
        "Attack precision",
        "Attack recall",
        "Macro-F1",
        "False-positive rate",
        "Missed attacks",
        "Training seconds",
        "Prediction seconds"
    ],
    "All-78": [
        78, accuracy, precision, recall,
        macro_f1, fpr, fn,
        rf_training_seconds, prediction_seconds
    ],
    "MI-30": [
        30, mi30_accuracy, mi30_precision,
        mi30_recall, mi30_macro_f1,
        mi30_fpr, fn30,
        mi30_training_seconds,
        mi30_prediction_seconds
    ]
})

print(comparison.to_string(index=False))

print("\nMI-30 confusion matrix:")
print(f"TN={tn30:,}  FP={fp30:,}")
print(f"FN={fn30:,}  TP={tp30:,}")

print("\nExperiment 43 completed.")


In [ ]:

import json
import pandas as pd

print("CICIDS2017 — Experiment 44")
print("Save All-78 vs MI-30 Comparison")
print("=" * 60)

output_folder = data_folder.parent / "experiment44"
output_folder.mkdir(parents=True, exist_ok=True)

results = pd.DataFrame([
    {
        "Model": "All-78",
        "Features": 78,
        "Accuracy": accuracy,
        "Attack_precision": precision,
        "Attack_recall": recall,
        "Macro_F1": macro_f1,
        "FPR": fpr,
        "False_positives": int(fp),
        "Missed_attacks": int(fn),
        "Training_seconds": rf_training_seconds,
        "Prediction_seconds": prediction_seconds
    },
    {
        "Model": "MI-30",
        "Features": 30,
        "Accuracy": mi30_accuracy,
        "Attack_precision": mi30_precision,
        "Attack_recall": mi30_recall,
        "Macro_F1": mi30_macro_f1,
        "FPR": mi30_fpr,
        "False_positives": int(fp30),
        "Missed_attacks": int(fn30),
        "Training_seconds": mi30_training_seconds,
        "Prediction_seconds": mi30_prediction_seconds
    }
])

csv_path = output_folder / "validation_model_comparison.csv"
json_path = output_folder / "validation_model_comparison.json"

results.to_csv(csv_path, index=False)

with open(json_path, "w") as f:
    json.dump(
        results.to_dict(orient="records"),
        f,
        indent=2
    )

print("CSV saved:", csv_path.exists())
print("JSON saved:", json_path.exists())

print("\nFeature reduction:",
      round((78 - 30) / 78 * 100, 2), "%")

print("Additional missed attacks with MI-30:",
      int(fn30 - fn))

print("Macro-F1 difference (MI-30 minus All-78):",
      round(mi30_macro_f1 - macro_f1, 6))

print("\nExperiment 44 completed.")


In [ ]:

import numpy as np
import time
import psutil
from sklearn.ensemble import RandomForestClassifier

print("CICIDS2017 — Experiment 45")
print("MI-50 Random Forest Training")
print("=" * 65)

# Use the existing training-only MI ranking
selected_features_50 = rankings[42][:50]

selected_indices_50 = [
    feature_names.index(f)
    for f in selected_features_50
]

# Select the top 50 features in their saved ranking order
X_train_50 = np.ascontiguousarray(
    X_train[:, selected_indices_50],
    dtype=np.float32
)

print("Training records:", f"{len(X_train_50):,}")
print("Selected features:", X_train_50.shape[1])

# Same hyperparameters as All-78 and MI-30
rf_mi50 = RandomForestClassifier(
    n_estimators=30,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=2
)

start = time.perf_counter()

rf_mi50.fit(X_train_50, y_train)

mi50_training_seconds = time.perf_counter() - start

print("\nTraining completed:", True)
print("Trees trained:", len(rf_mi50.estimators_))
print(
    "Training time (seconds):",
    round(mi50_training_seconds, 2)
)

print(
    "Available RAM after training:",
    round(
        psutil.virtual_memory().available / (1024**3),
        2
    ),
    "GB"
)

print("\nExperiment 45 completed.")


In [ ]:

import numpy as np
import pandas as pd
import time

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("CICIDS2017 — Experiment 46")
print("All-78 vs MI-30 vs MI-50 Validation")
print("=" * 65)

# Same validation records, using the MI-50 feature order
X_val_50 = np.ascontiguousarray(
    X_val[:, selected_indices_50],
    dtype=np.float32
)

start = time.perf_counter()
y_pred_50 = rf_mi50.predict(X_val_50)
mi50_prediction_seconds = time.perf_counter() - start

tn50, fp50, fn50, tp50 = confusion_matrix(
    y_val, y_pred_50, labels=[0, 1]
).ravel()

mi50_accuracy = accuracy_score(y_val, y_pred_50)
mi50_precision = precision_score(
    y_val, y_pred_50, zero_division=0
)
mi50_recall = recall_score(
    y_val, y_pred_50, zero_division=0
)
mi50_macro_f1 = f1_score(
    y_val, y_pred_50,
    average="macro",
    zero_division=0
)
mi50_fpr = fp50 / (fp50 + tn50)

comparison_3 = pd.DataFrame({
    "Metric": [
        "Features",
        "Accuracy",
        "Attack precision",
        "Attack recall",
        "Macro-F1",
        "False-positive rate",
        "Missed attacks",
        "False alarms",
        "Training seconds",
        "Prediction seconds"
    ],
    "All-78": [
        78, accuracy, precision, recall,
        macro_f1, fpr, fn, fp,
        rf_training_seconds, prediction_seconds
    ],
    "MI-30": [
        30, mi30_accuracy, mi30_precision,
        mi30_recall, mi30_macro_f1,
        mi30_fpr, fn30, fp30,
        mi30_training_seconds,
        mi30_prediction_seconds
    ],
    "MI-50": [
        50, mi50_accuracy, mi50_precision,
        mi50_recall, mi50_macro_f1,
        mi50_fpr, fn50, fp50,
        mi50_training_seconds,
        mi50_prediction_seconds
    ]
})

print(comparison_3.to_string(index=False))

print("\nMI-50 confusion matrix:")
print(f"TN={tn50:,}  FP={fp50:,}")
print(f"FN={fn50:,}  TP={tp50:,}")

print("\nExperiment 46 completed.")


In [ ]:

import pandas as pd
import json

print("CICIDS2017 — Experiment 47")
print("Save Three-Model Validation Comparison")
print("=" * 60)

output_folder = data_folder.parent / "experiment47"
output_folder.mkdir(parents=True, exist_ok=True)

results = pd.DataFrame([
    {
        "Model": "All-78",
        "Features": 78,
        "Accuracy": float(accuracy),
        "Attack_precision": float(precision),
        "Attack_recall": float(recall),
        "Macro_F1": float(macro_f1),
        "FPR": float(fpr),
        "Missed_attacks": int(fn),
        "False_alarms": int(fp),
        "Training_seconds": float(rf_training_seconds),
        "Prediction_seconds": float(prediction_seconds)
    },
    {
        "Model": "MI-30",
        "Features": 30,
        "Accuracy": float(mi30_accuracy),
        "Attack_precision": float(mi30_precision),
        "Attack_recall": float(mi30_recall),
        "Macro_F1": float(mi30_macro_f1),
        "FPR": float(mi30_fpr),
        "Missed_attacks": int(fn30),
        "False_alarms": int(fp30),
        "Training_seconds": float(mi30_training_seconds),
        "Prediction_seconds": float(mi30_prediction_seconds)
    },
    {
        "Model": "MI-50",
        "Features": 50,
        "Accuracy": float(mi50_accuracy),
        "Attack_precision": float(mi50_precision),
        "Attack_recall": float(mi50_recall),
        "Macro_F1": float(mi50_macro_f1),
        "FPR": float(mi50_fpr),
        "Missed_attacks": int(fn50),
        "False_alarms": int(fp50),
        "Training_seconds": float(mi50_training_seconds),
        "Prediction_seconds": float(mi50_prediction_seconds)
    }
])

csv_path = output_folder / "three_model_validation.csv"
json_path = output_folder / "three_model_validation.json"

results.to_csv(csv_path, index=False)

with open(json_path, "w") as f:
    json.dump(
        results.to_dict(orient="records"),
        f,
        indent=2
    )

print("CSV saved:", csv_path.exists())
print("JSON saved:", json_path.exists())

print("\nModel summary:")
print(
    results[
        ["Model", "Features", "Macro_F1",
         "FPR", "Missed_attacks"]
    ].to_string(index=False)
)

print("\nExperiment 47 completed.")


In [ ]:

import joblib
import json
from pathlib import Path

print("CICIDS2017 — Experiment 48")
print("Save Trained Random Forest Models")
print("=" * 60)

output_folder = data_folder.parent / "experiment48"
output_folder.mkdir(parents=True, exist_ok=True)

models = {
    "all78": rf_baseline,
    "mi30": rf_mi30,
    "mi50": rf_mi50
}

for name, model in models.items():
    path = output_folder / f"rf_{name}.joblib"
    joblib.dump(model, path, compress=3)

    print(
        f"{name}: saved={path.exists()}, "
        f"size={path.stat().st_size / (1024**2):.2f} MB"
    )

# Save feature order for reproducible predictions
feature_metadata = {
    "all78": list(feature_names),
    "mi30": list(selected_features_30),
    "mi50": list(selected_features_50),
    "label_definition": {
        "0": "BENIGN",
        "1": "Attack"
    },
    "training_partition": "Frozen duplicate-aware training split",
    "validation_partition": "Frozen duplicate-aware validation split"
}

metadata_path = output_folder / "model_feature_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(feature_metadata, f, indent=2)

print("Feature metadata saved:", metadata_path.exists())
print("\nExperiment 48 completed.")


In [ ]:

import joblib
import json
import numpy as np

print("CICIDS2017 — Experiment 49")
print("Saved Model Reload Verification")
print("=" * 60)

model_folder = data_folder.parent / "experiment48"

with open(
    model_folder / "model_feature_metadata.json", "r"
) as f:
    saved_metadata = json.load(f)

model_specs = {
    "all78": (rf_baseline, X_val),
    "mi30": (rf_mi30, X_val_30),
    "mi50": (rf_mi50, X_val_50)
}

all_checks_passed = True

for name, (original_model, features) in model_specs.items():

    saved_path = model_folder / f"rf_{name}.joblib"
    reloaded_model = joblib.load(saved_path)

    # Use only 1,000 validation records for verification
    sample = features[:1000]

    original_predictions = original_model.predict(sample)
    reloaded_predictions = reloaded_model.predict(sample)

    predictions_match = np.array_equal(
        original_predictions,
        reloaded_predictions
    )

    feature_count_matches = (
        reloaded_model.n_features_in_
        == len(saved_metadata[name])
    )

    passed = predictions_match and feature_count_matches
    all_checks_passed &= passed

    print(
        f"{name}: "
        f"predictions_match={predictions_match}, "
        f"feature_count_matches={feature_count_matches}"
    )

print("\nAll checks passed:", all_checks_passed)
print("\nExperiment 49 completed.")


In [ ]:

import pandas as pd
import numpy as np

print("CICIDS2017 — Experiment 50")
print("Validation Detection by Attack Category")
print("=" * 65)

# The original multiclass labels are preserved in val_df.
actual_categories = (
    val_df["Label"]
    .astype(str)
    .str.strip()
    .to_numpy()
)

# Reuse All-78 validation predictions from Experiment 36
category_results = pd.DataFrame({
    "Attack_category": actual_categories,
    "Predicted_attack": y_pred.astype(int)
})

# Exclude BENIGN: examine detection of attack categories
attack_results = category_results.loc[
    category_results["Attack_category"] != "BENIGN"
].copy()

summary = (
    attack_results
    .groupby("Attack_category")
    .agg(
        Total_attacks=("Predicted_attack", "size"),
        Detected=("Predicted_attack", "sum")
    )
    .reset_index()
)

summary["Missed"] = (
    summary["Total_attacks"] - summary["Detected"]
)

summary["Recall"] = (
    summary["Detected"] / summary["Total_attacks"]
)

summary = summary.sort_values(
    ["Recall", "Total_attacks"],
    ascending=[True, False]
)

print(summary.to_string(index=False))

print("\nTotal attacks:", summary["Total_attacks"].sum())
print("Total missed:", summary["Missed"].sum())
print("Expected missed attacks:", int(fn))

print("\nExperiment 50 completed.")


In [ ]:

import pandas as pd
import numpy as np

print("CICIDS2017 — Experiment 51")
print("Per-Category Recall Across Three Models")
print("=" * 65)

actual_categories = (
    val_df["Label"].astype(str).str.strip().to_numpy()
)

attack_mask = actual_categories != "BENIGN"

category_comparison = pd.DataFrame({
    "Attack_category": actual_categories[attack_mask],
    "All78_detected": y_pred[attack_mask],
    "MI30_detected": y_pred_30[attack_mask],
    "MI50_detected": y_pred_50[attack_mask]
})

summary_51 = (
    category_comparison
    .groupby("Attack_category")
    .agg(
        Total=("All78_detected", "size"),
        All78_detected=("All78_detected", "sum"),
        MI30_detected=("MI30_detected", "sum"),
        MI50_detected=("MI50_detected", "sum")
    )
    .reset_index()
)

for model in ["All78", "MI30", "MI50"]:
    summary_51[f"{model}_missed"] = (
        summary_51["Total"]
        - summary_51[f"{model}_detected"]
    )

    summary_51[f"{model}_recall"] = (
        summary_51[f"{model}_detected"]
        / summary_51["Total"]
    )

display_columns = [
    "Attack_category", "Total",
    "All78_missed", "MI30_missed", "MI50_missed",
    "All78_recall", "MI30_recall", "MI50_recall"
]

print(
    summary_51[display_columns]
    .sort_values("All78_recall")
    .to_string(index=False)
)

print("\nTotal missed attacks:")
for model in ["All78", "MI30", "MI50"]:
    print(
        model,
        int(summary_51[f"{model}_missed"].sum())
    )

print("\nExperiment 51 completed.")


In [ ]:

import pandas as pd
import json

print("CICIDS2017 — Experiment 52")
print("Save Per-Category Detection Evidence")
print("=" * 60)

output_folder = data_folder.parent / "experiment52"
output_folder.mkdir(parents=True, exist_ok=True)

csv_path = output_folder / "category_recall_comparison.csv"
json_path = output_folder / "category_recall_comparison.json"

# Save the complete table from Experiment 51
summary_51.to_csv(csv_path, index=False)

with open(json_path, "w") as f:
    json.dump(
        summary_51.to_dict(orient="records"),
        f,
        indent=2
    )

# Focus on two categories with substantial recall losses
focus_categories = [
    "Web Attack � Brute Force",
    "Web Attack � XSS"
]

# Use keyword matching because CICIDS2017 label
# strings sometimes contain encoding artifacts.
focus = summary_51.loc[
    summary_51["Attack_category"].str.contains(
        "Brute Force|XSS",
        regex=True,
        na=False
    )
]

print("CSV saved:", csv_path.exists())
print("JSON saved:", json_path.exists())

print("\nWeb attack comparison:")
print(
    focus[
        [
            "Attack_category",
            "Total",
            "All78_missed",
            "MI30_missed",
            "MI50_missed"
        ]
    ].to_string(index=False)
)

print("\nExperiment 52 completed.")


In [ ]:

import pandas as pd

print("CICIDS2017 — Experiment 53")
print("MI Feature Retention and Removal Audit")
print("=" * 65)

all_features = list(feature_names)

mi30_set = set(selected_features_30)
mi50_set = set(selected_features_50)

# Features excluded from each reduced model
removed_mi30 = [
    f for f in all_features if f not in mi30_set
]

removed_mi50 = [
    f for f in all_features if f not in mi50_set
]

# Features present in MI-50 but absent from MI-30
extra_mi50 = [
    f for f in selected_features_50
    if f not in mi30_set
]

print("Total original features:", len(all_features))
print("MI-30 retained:", len(mi30_set))
print("MI-30 removed:", len(removed_mi30))
print("MI-50 retained:", len(mi50_set))
print("MI-50 removed:", len(removed_mi50))

print("\nFeatures added by MI-50 compared with MI-30:")
for i, feature in enumerate(extra_mi50, 1):
    print(f"{i:2d}. {feature}")

print("\nFeatures excluded by MI-50:")
for i, feature in enumerate(removed_mi50, 1):
    print(f"{i:2d}. {feature}")

print("\nFeatures excluded by MI-30:")
for i, feature in enumerate(removed_mi30, 1):
    print(f"{i:2d}. {feature}")

print("\nExperiment 53 completed.")


In [ ]:
import pandas as pd

print("CICIDS2017 — Experiment 54")
print("MI Scores of Retained and Removed Features")
print("=" * 65)

# Use the existing MI ranking from Experiment 39.
# Do not recalculate MI or access validation/test labels.
audit = mi_ranking.copy()

print("Available columns:", audit.columns.tolist())

# Identify the existing feature and score columns
feature_col = "Feature"
score_col = "MI_Score"

assert feature_col in audit.columns
assert score_col in audit.columns

audit["Rank"] = range(1, len(audit) + 1)

audit["MI30_status"] = audit[feature_col].apply(
    lambda f: "Retained" if f in mi30_set else "Removed"
)

audit["MI50_status"] = audit[feature_col].apply(
    lambda f: "Retained" if f in mi50_set else "Removed"
)

print("\nFeatures ranked 25–55:")
print(
    audit.loc[
        audit["Rank"].between(25, 55),
        ["Rank", feature_col, score_col,
         "MI30_status", "MI50_status"]
    ].to_string(index=False)
)

print("\nMI score summaries:")
print(
    audit.groupby("MI30_status")[score_col]
    .agg(["count", "min", "median", "max"])
    .to_string()
)

print("\nExperiment 54 completed.")


In [ ]:
import numpy as np
import pandas as pd

print("CICIDS2017 — Experiment 55")
print("MI Ranking Order Verification")
print("=" * 60)

# Work on a copy of the original ranking
audit_55 = mi_ranking.copy().reset_index(drop=True)

scores = audit_55["MI_Score"].to_numpy()

# A correctly descending ranking should never increase
increases = np.where(np.diff(scores) > 1e-12)[0]

print("Features ranked:", len(audit_55))
print("Score increases found:", len(increases))

print("\nFirst score-order violations:")
for i in increases[:15]:
    print(
        f"Rank {i+1}: "
        f"{audit_55.loc[i, 'Feature']} "
        f"({scores[i]:.6f})"
    )
    print(
        f"Rank {i+2}: "
        f"{audit_55.loc[i+1, 'Feature']} "
        f"({scores[i+1]:.6f})"
    )

# Compare saved top-30 against the top-30 by MI score
score_sorted = audit_55.sort_values(
    "MI_Score",
    ascending=False,
    kind="stable"
)

saved_top30 = set(selected_features_30)
score_top30 = set(
    score_sorted.head(30)["Feature"]
)

print("\nSaved MI-30 equals score-sorted MI-30:",
      saved_top30 == score_top30)

print("Only in saved MI-30:",
      sorted(saved_top30 - score_top30))

print("Only in score-sorted MI-30:",
      sorted(score_top30 - saved_top30))

print("\nExperiment 55 completed.")


In [ ]:
import pandas as pd

print("CICIDS2017 — Experiment 56")
print("Random Forest Importance vs MI Selection")
print("=" * 65)

importance_56 = pd.DataFrame({
    "Feature": feature_names,
    "RF_importance": rf_baseline.feature_importances_
})

importance_56["MI30_status"] = importance_56[
    "Feature"
].apply(
    lambda f: "Retained" if f in mi30_set else "Removed"
)

importance_56["MI50_status"] = importance_56[
    "Feature"
].apply(
    lambda f: "Retained" if f in mi50_set else "Removed"
)

importance_56 = importance_56.merge(
    mi_ranking[["Feature", "MI_Score"]],
    on="Feature",
    how="left"
)

importance_56 = importance_56.sort_values(
    "RF_importance",
    ascending=False
).reset_index(drop=True)

print("\nTop 25 features by Random Forest importance:")
print(
    importance_56.head(25)[
        [
            "Feature",
            "RF_importance",
            "MI_Score",
            "MI30_status",
            "MI50_status"
        ]
    ].to_string(index=False)
)

print("\nTotal RF importance assigned to:")
print(
    importance_56.groupby("MI30_status")[
        "RF_importance"
    ].sum()
)

print("\nExperiment 56 completed.")


In [ ]:
import json
import pandas as pd

print("CICIDS2017 — Experiment 57")
print("Save MI and RF Feature Importance Audit")
print("=" * 60)

output_folder = data_folder.parent / "experiment57"
output_folder.mkdir(parents=True, exist_ok=True)

csv_path = output_folder / "mi_vs_rf_importance.csv"
json_path = output_folder / "mi_vs_rf_importance_summary.json"

# Save all 78 feature-level results
importance_56.to_csv(csv_path, index=False)

removed_importance = float(
    importance_56.loc[
        importance_56["MI30_status"] == "Removed",
        "RF_importance"
    ].sum()
)

retained_importance = float(
    importance_56.loc[
        importance_56["MI30_status"] == "Retained",
        "RF_importance"
    ].sum()
)

summary = {
    "dataset": "CICIDS2017",
    "model": "All-78 Random Forest",
    "importance_method": "Mean decrease in impurity",
    "features_total": 78,
    "mi30_features_retained": 30,
    "mi30_features_removed": 48,
    "rf_importance_retained": retained_importance,
    "rf_importance_removed": removed_importance,
    "interpretation_limit": (
        "Impurity importance is not causal and does not "
        "measure attack-category-specific performance."
    )
}

with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

print("CSV saved:", csv_path.exists())
print("JSON saved:", json_path.exists())

print("Retained RF importance:",
      round(retained_importance, 6))
print("Removed RF importance:",
      round(removed_importance, 6))

print("\nExperiment 57 completed.")


In [ ]:

import numpy as np
import pandas as pd

print("CICIDS2017 — Experiment 58")
print("All-78 vs MI-30 Attack Detection Disagreements")
print("=" * 65)

categories = (
    val_df["Label"]
    .astype(str)
    .str.strip()
    .to_numpy()
)

# Only actual attack records
attack_mask = categories != "BENIGN"

disagreements = pd.DataFrame({
    "Category": categories[attack_mask],
    "All78": np.asarray(y_pred)[attack_mask],
    "MI30": np.asarray(y_pred_30)[attack_mask]
})

# All-78 detects, MI-30 misses
disagreements["All78_only"] = (
    (disagreements["All78"] == 1) &
    (disagreements["MI30"] == 0)
)

# MI-30 detects, All-78 misses
disagreements["MI30_only"] = (
    (disagreements["All78"] == 0) &
    (disagreements["MI30"] == 1)
)

# Both miss
disagreements["Both_miss"] = (
    (disagreements["All78"] == 0) &
    (disagreements["MI30"] == 0)
)

summary_58 = (
    disagreements.groupby("Category")
    .agg(
        Total=("All78", "size"),
        All78_only=("All78_only", "sum"),
        MI30_only=("MI30_only", "sum"),
        Both_miss=("Both_miss", "sum")
    )
    .reset_index()
)

summary_58 = summary_58.sort_values(
    "All78_only",
    ascending=False
)

print(summary_58.to_string(index=False))

print("\nOverall disagreements:")
print("Detected only by All-78:",
      int(disagreements["All78_only"].sum()))
print("Detected only by MI-30:",
      int(disagreements["MI30_only"].sum()))
print("Missed by both:",
      int(disagreements["Both_miss"].sum()))

print("\nConsistency checks:")
all78_missed = (
    disagreements["MI30_only"].sum()
    + disagreements["Both_miss"].sum()
)

mi30_missed = (
    disagreements["All78_only"].sum()
    + disagreements["Both_miss"].sum()
)

print("All-78 missed:", int(all78_missed))
print("MI-30 missed:", int(mi30_missed))

assert all78_missed == 330
assert mi30_missed == 567

print("\nExperiment 58 completed.")


In [ ]:

import json

print("CICIDS2017 — Experiment 59")
print("Save Model Disagreement Evidence")
print("=" * 60)

output_folder = data_folder.parent / "experiment59"
output_folder.mkdir(parents=True, exist_ok=True)

csv_path = output_folder / "attack_disagreements_by_category.csv"
json_path = output_folder / "attack_disagreement_summary.json"

# Save the full per-category comparison
summary_58.to_csv(csv_path, index=False)

all78_only = int(disagreements["All78_only"].sum())
mi30_only = int(disagreements["MI30_only"].sum())
both_miss = int(disagreements["Both_miss"].sum())

web_rows = summary_58[
    summary_58["Category"].str.contains(
        "Brute Force|XSS",
        regex=True,
        na=False
    )
]

web_all78_only = int(web_rows["All78_only"].sum())

evidence = {
    "dataset": "CICIDS2017",
    "partition": "Frozen duplicate-aware validation",
    "comparison": "All-78 vs MI-30",
    "all78_only_detected": all78_only,
    "mi30_only_detected": mi30_only,
    "missed_by_both": both_miss,
    "all78_total_missed": int(mi30_only + both_miss),
    "mi30_total_missed": int(all78_only + both_miss),
    "additional_missed_with_mi30": int(
        all78_only - mi30_only
    ),
    "web_attacks_detected_only_by_all78": web_all78_only,
    "web_share_of_all78_only_percent": round(
        100 * web_all78_only / all78_only, 2
    ),
    "interpretation_limit": (
        "Model disagreements do not establish which "
        "removed features caused the detection differences."
    )
}

with open(json_path, "w") as f:
    json.dump(evidence, f, indent=2)

print("CSV saved:", csv_path.exists())
print("JSON saved:", json_path.exists())
print("All-78 only:", all78_only)
print("MI-30 only:", mi30_only)
print("Both miss:", both_miss)
print("Web attacks detected only by All-78:",
      web_all78_only)
print("Web share (%):",
      evidence["web_share_of_all78_only_percent"])

print("\nExperiment 59 completed.")


In [ ]:

import numpy as np
import pandas as pd

print("CICIDS2017 — Experiment 60")
print("Web-Attack Disagreement Source Audit")
print("=" * 65)

# Confirm that the validation DataFrame contains
# the source-file information needed for this audit.
print("Validation columns:")
print(val_df.columns.tolist())

source_candidates = [
    "source_file",
    "Source_File",
    "source",
    "filename",
    "file"
]

source_col = next(
    (c for c in source_candidates if c in val_df.columns),
    None
)

print("\nSource column found:", source_col)

categories = (
    val_df["Label"].astype(str).str.strip().to_numpy()
)

web_mask = (
    pd.Series(categories)
    .str.contains("Brute Force|XSS", regex=True, na=False)
    .to_numpy()
)

web_audit = pd.DataFrame({
    "Category": categories[web_mask],
    "All78_detected": np.asarray(y_pred)[web_mask],
    "MI30_detected": np.asarray(y_pred_30)[web_mask]
})

web_audit["All78_only"] = (
    (web_audit["All78_detected"] == 1)
    & (web_audit["MI30_detected"] == 0)
)

if source_col is not None:
    web_audit["Source_file"] = (
        val_df[source_col].astype(str).to_numpy()[web_mask]
    )

    result = (
        web_audit.groupby(["Source_file", "Category"])
        .agg(
            Records=("All78_only", "size"),
            All78_only=("All78_only", "sum")
        )
        .reset_index()
    )

    print("\nWeb-attack disagreements by source:")
    print(result.to_string(index=False))

else:
    print(
        "\nSource-file column is not present in val_df."
    )
    print(
        "We can use the frozen split-assignment file "
        "for a separate source audit if needed."
    )

print("\nTotal web-attack records:", len(web_audit))
print(
    "Web attacks detected only by All-78:",
    int(web_audit["All78_only"].sum())
)

assert len(web_audit) == 324
assert int(web_audit["All78_only"].sum()) == 257

print("\nExperiment 60 completed.")


In [ ]:

import pandas as pd
from pathlib import Path

print("CICIDS2017 — Experiment 61")
print("Frozen Split Metadata Inspection")
print("=" * 65)

split_file = (
    data_folder.parent
    / "experiment16"
    / "provisional_split_assignments.csv"
)

print("Split file exists:", split_file.exists())

if split_file.exists():
    preview = pd.read_csv(split_file, nrows=5)

    print("\nAvailable columns:")
    print(preview.columns.tolist())

    print("\nFirst five records:")
    print(preview.to_string(index=False))

    print("\nFile size (MB):",
          round(split_file.stat().st_size / (1024**2), 2))

else:
    print("The split file was not found at the expected path.")

print("\nExperiment 61 completed.")


In [ ]:

import pandas as pd

print("CICIDS2017 — Experiment 62")
print("Verify Web-Attack Source Files")
print("=" * 65)

# Read only the metadata columns we need
assignments = pd.read_csv(
    split_file,
    usecols=["source_file", "row_number", "partition"]
)

# Restrict to validation assignments
val_assignments = assignments.loc[
    assignments["partition"] == "Validation"
].copy()

del assignments

results_62 = []

for file in csv_files:
    # Read only the original attack labels
    labels = pd.read_csv(
        file,
        usecols=lambda c: c.strip() == "Label",
        low_memory=False
    )

    labels.columns = labels.columns.str.strip()

    file_assignments = val_assignments.loc[
        val_assignments["source_file"] == file.name,
        ["row_number"]
    ]

    # Original row_number values are zero-based
    # as recorded in the frozen assignment file.
    selected_rows = file_assignments["row_number"].to_numpy()

    selected_labels = (
        labels.iloc[selected_rows]["Label"]
        .astype(str)
        .str.strip()
    )

    web_labels = selected_labels[
        selected_labels.str.contains(
            "Brute Force|XSS",
            regex=True,
            na=False
        )
    ]

    results_62.append({
        "Source_file": file.name,
        "Validation_web_attacks": len(web_labels),
        "Brute_Force": int(
            web_labels.str.contains("Brute Force").sum()
        ),
        "XSS": int(
            web_labels.str.contains("XSS").sum()
        )
    })

source_summary_62 = pd.DataFrame(results_62)

print(
    source_summary_62.to_string(index=False)
)

print("\nTotals:")
print(
    source_summary_62[
        ["Validation_web_attacks", "Brute_Force", "XSS"]
    ].sum()
)

print("\nExperiment 62 completed.")


In [ ]:
import json

print("CICIDS2017 — Experiment 63")
print("Save Web-Attack Source Verification")
print("=" * 60)

output_folder = data_folder.parent / "experiment63"
output_folder.mkdir(parents=True, exist_ok=True)

csv_path = output_folder / "web_attack_source_distribution.csv"
json_path = output_folder / "web_attack_source_audit.json"

source_summary_62.to_csv(csv_path, index=False)

web_source = source_summary_62.loc[
    source_summary_62["Validation_web_attacks"] > 0
]

evidence = {
    "dataset": "CICIDS2017",
    "partition": "Frozen duplicate-aware validation",
    "web_attack_records": 324,
    "brute_force_records": 230,
    "xss_records": 94,
    "source_files_with_web_attacks": (
        web_source["Source_file"].tolist()
    ),
    "source_file_count": len(web_source),
    "limitation": (
        "Duplicate-aware splitting prevents exact "
        "feature-pattern overlap but does not ensure "
        "independence between capture sessions or days."
    )
}

with open(json_path, "w") as f:
    json.dump(evidence, f, indent=2)

print("CSV saved:", csv_path.exists())
print("JSON saved:", json_path.exists())
print("Source files containing web attacks:",
      evidence["source_file_count"])
print("Brute Force records:",
      evidence["brute_force_records"])
print("XSS records:",
      evidence["xss_records"])

assert len(web_source) == 1
assert int(source_summary_62["Brute_Force"].sum()) == 230
assert int(source_summary_62["XSS"].sum()) == 94

print("\nExperiment 63 completed.")


In [ ]:

import pandas as pd
from pathlib import Path

print("CICIDS2017 — Experiment 64")
print("Frozen Test Partition Integrity Check")
print("=" * 65)

# Use the existing frozen split-assignment file
split_file = (
    data_folder.parent
    / "experiment16"
    / "provisional_split_assignments.csv"
)

expected_counts = {
    "Train": 1_988_583,
    "Validation": 420_601,
    "Test": 421_559
}

assert split_file.exists(), "Frozen split file not found."

# Read only the metadata needed for verification
assignments_64 = pd.read_csv(
    split_file,
    usecols=[
        "source_file",
        "row_number",
        "feature_hash",
        "partition"
    ],
    dtype={
        "source_file": "string",
        "row_number": "int64",
        "feature_hash": "string",
        "partition": "string"
    }
)

print("\nPartition counts:")
counts = assignments_64["partition"].value_counts()

for partition, expected in expected_counts.items():
    actual = int(counts.get(partition, 0))
    print(f"{partition}: {actual:,} (expected {expected:,})")
    assert actual == expected, f"{partition} count mismatch"

assert len(assignments_64) == 2_830_743
assert set(counts.index) == set(expected_counts)

print("\nMetadata checks:")

missing = int(assignments_64.isna().sum().sum())

duplicate_ids = int(
    assignments_64.duplicated(
        subset=["source_file", "row_number"]
    ).sum()
)

hash_partition_counts = (
    assignments_64
    .groupby("feature_hash")["partition"]
    .nunique()
)

cross_partition_hashes = int(
    (hash_partition_counts > 1).sum()
)

print("Missing metadata values:", missing)
print("Duplicate source-file/row-number pairs:", duplicate_ids)
print("Feature hashes crossing partitions:",
      cross_partition_hashes)

assert missing == 0
assert duplicate_ids == 0
assert cross_partition_hashes == 0

print("\nFrozen test partition records:",
      int(counts["Test"]))

print("All integrity checks passed: True")
print("\nExperiment 64 completed.")


In [ ]:
from pathlib import Path
import json

print("CICIDS2017 — Experiment 65")
print("Test Preparation Readiness Check")
print("=" * 60)

research_folder = data_folder.parent

# Training-derived imputation parameters
medians_file = (
    research_folder
    / "experiment25"
    / "training_medians.json"
)

# Previously saved models
models_folder = research_folder / "experiment48"

print("Training medians file exists:",
      medians_file.exists())

if medians_file.exists():
    with open(medians_file, "r") as f:
        medians = json.load(f)

    print("Training median parameter names:",
          list(medians.keys()))

print("\nExperiment 48 folder exists:",
      models_folder.exists())

if models_folder.exists():
    print("Saved files:")
    for file in sorted(models_folder.iterdir()):
        if file.is_file():
            print(" -", file.name)

print("\nFrozen test records:",
      int(counts["Test"]))

print("\nExperiment 65 completed.")


In [ ]:
import json

print("CICIDS2017 — Experiment 66")
print("Saved Model Feature Metadata Inspection")
print("=" * 65)

metadata_path = (
    data_folder.parent
    / "experiment48"
    / "model_feature_metadata.json"
)

assert metadata_path.exists(), "Model metadata file not found."

with open(metadata_path, "r") as f:
    model_metadata = json.load(f)

print("Metadata type:", type(model_metadata).__name__)

if isinstance(model_metadata, dict):
    print("\nTop-level metadata keys:")
    for key in model_metadata:
        print(" -", key)

    print("\nFeature metadata details:")
    for key, value in model_metadata.items():
        if isinstance(value, list):
            print(f"\n{key}: {len(value)} entries")
            print("First five:", value[:5])
            print("Last five:", value[-5:])
        elif isinstance(value, dict):
            print(f"\n{key}:")
            print("Nested keys:", list(value.keys()))
        else:
            print(f"{key}: {value}")

print("\nExperiment 66 completed.")


In [ ]:

import pandas as pd
import numpy as np
import json
from pathlib import Path

print("CICIDS2017 — Experiment 67")
print("Frozen Test Dataset Export and Imputation")
print("=" * 65)

research_folder = data_folder.parent
output_folder = research_folder / "experiment67"
output_folder.mkdir(parents=True, exist_ok=True)

test_path = output_folder / "test_clean.csv"

# Load training-derived medians only
with open(
    research_folder / "experiment25" / "training_medians.json",
    "r"
) as f:
    train_medians = json.load(f)

# Read frozen test assignments
test_assignments = pd.read_csv(
    split_file,
    usecols=["source_file", "row_number", "partition"]
)

test_assignments = test_assignments.loc[
    test_assignments["partition"] == "Test",
    ["source_file", "row_number"]
]

assert len(test_assignments) == 421_559

# Save to a temporary file until all checks pass
temp_path = output_folder / "test_clean_incomplete.csv"
if temp_path.exists():
    temp_path.unlink()

total_saved = 0
write_header = True

for file in csv_files:
    selected_rows = set(
        test_assignments.loc[
            test_assignments["source_file"] == file.name,
            "row_number"
        ].astype(int)
    )

    file_saved = 0
    row_offset = 0

    for chunk in pd.read_csv(
        file,
        chunksize=50_000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip()

        # Match original zero-based row positions
        positions = np.arange(
            row_offset,
            row_offset + len(chunk)
        )

        mask = np.isin(
            positions,
            list(selected_rows)
        )

        selected = chunk.loc[mask].copy()
        row_offset += len(chunk)

        if selected.empty:
            continue

        # Replace infinity and impute using training medians
        for column, median in train_medians.items():
            assert column in selected.columns, (
                f"Missing column: {column}"
            )

            selected[column] = pd.to_numeric(
                selected[column],
                errors="coerce"
            )

            selected[column] = (
                selected[column]
                .replace([np.inf, -np.inf], np.nan)
                .fillna(float(median))
            )

        # Preserve the original feature and label columns
        selected.to_csv(
            temp_path,
            mode="w" if write_header else "a",
            header=write_header,
            index=False
        )

        write_header = False
        file_saved += len(selected)
        total_saved += len(selected)

    print(f"Processed: {file.name} | Test rows: {file_saved:,}")

print("\nTotal test records saved:", f"{total_saved:,}")

assert total_saved == 421_559, (
    "Test row count mismatch — do not use this file."
)

# Rename only after the record-count check succeeds
temp_path.replace(test_path)

print("Test file exists:", test_path.exists())
print("Output:", test_path)
print("\nExperiment 67 completed.")


In [ ]:
import pandas as pd
import numpy as np

print("CICIDS2017 — Experiment 68")
print("Saved Test Dataset Integrity Check")
print("=" * 65)

test_path = (
    data_folder.parent
    / "experiment67"
    / "test_clean.csv"
)

assert test_path.exists(), "Test file not found."

expected_features = model_metadata["all78"]
expected_columns = expected_features + ["Label"]

total_rows = 0
missing_values = 0
infinite_values = 0
columns_match = True
unexpected_labels = 0

for chunk in pd.read_csv(
    test_path,
    chunksize=50_000,
    low_memory=False
):
    chunk.columns = chunk.columns.str.strip()

    total_rows += len(chunk)

    if chunk.columns.tolist() != expected_columns:
        columns_match = False

    features = chunk[expected_features].apply(
        pd.to_numeric,
        errors="coerce"
    )

    missing_values += int(
        features.isna().sum().sum()
    )

    infinite_values += int(
        np.isinf(features.to_numpy()).sum()
    )

    # Check label format only; do not report class counts
    labels = chunk["Label"].astype(str).str.strip()

    unexpected_labels += int(
        (~labels.isin(["BENIGN", "Bot", "DDoS",
                      "PortScan", "Infiltration",
                      "Heartbleed", "FTP-Patator",
                      "SSH-Patator", "DoS Hulk",
                      "DoS GoldenEye",
                      "DoS slowloris",
                      "DoS Slowhttptest",
                      "Web Attack � Brute Force",
                      "Web Attack � XSS",
                      "Web Attack � Sql Injection"]
                     )).sum()
    )

print("Test records:", f"{total_rows:,}")
print("Expected records:", "421,559")
print("Feature columns:", len(expected_features))
print("Column order matches:", columns_match)
print("Missing feature values:", missing_values)
print("Infinite feature values:", infinite_values)
print("Unexpected label strings:", unexpected_labels)

assert total_rows == 421_559
assert columns_match
assert missing_values == 0
assert infinite_values == 0
assert unexpected_labels == 0

print("\nAll integrity checks passed: True")
print("Experiment 68 completed.")


In [ ]:
import joblib

print("CICIDS2017 — Experiment 69")
print("Final Model Readiness Verification")
print("=" * 65)

models_folder = data_folder.parent / "experiment48"

model_files = {
    "All-78": ("rf_all78.joblib", "all78", 78),
    "MI-30": ("rf_mi30.joblib", "mi30", 30),
    "MI-50": ("rf_mi50.joblib", "mi50", 50)
}

final_models = {}

for name, (filename, metadata_key, expected_n) in model_files.items():
    model_path = models_folder / filename

    assert model_path.exists(), f"Missing model: {filename}"

    model = joblib.load(model_path)
    saved_features = model_metadata[metadata_key]

    assert len(saved_features) == expected_n
    assert model.n_features_in_ == expected_n

    final_models[name] = {
        "model": model,
        "features": saved_features
    }

    print(f"\n{name}")
    print("Model loaded: True")
    print("Expected features:", expected_n)
    print("Model input features:", model.n_features_in_)
    print("Saved feature names:", len(saved_features))
    print("Trees:", model.n_estimators)

print("\nAll three models ready: True")
print("Experiment 69 completed.")


In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("CICIDS2017 — Experiment 70")
print("FINAL HELD-OUT TEST EVALUATION")
print("=" * 65)

test_path = (
    data_folder.parent
    / "experiment67"
    / "test_clean.csv"
)

# Keep model order fixed
model_names = ["All-78", "MI-30", "MI-50"]

# Accumulate predictions in the original test-file order
predictions_70 = {name: [] for name in model_names}
true_labels_70 = []

prediction_times_70 = {
    name: 0.0 for name in model_names
}

for chunk_number, chunk in enumerate(
    pd.read_csv(test_path, chunksize=50_000),
    start=1
):
    chunk.columns = chunk.columns.str.strip()

    # Binary ground truth: 0 = BENIGN, 1 = attack
    y_chunk = (
        chunk["Label"]
        .astype(str)
        .str.strip()
        .ne("BENIGN")
        .astype(np.int8)
        .to_numpy()
    )

    true_labels_70.append(y_chunk)

    for name in model_names:
        model = final_models[name]["model"]
        features = final_models[name]["features"]

        X_chunk = chunk.loc[:, features].astype(
            np.float32
        )

        start = time.perf_counter()
        predicted = model.predict(X_chunk)
        prediction_times_70[name] += (
            time.perf_counter() - start
        )

        predictions_70[name].append(
            np.asarray(predicted, dtype=np.int8)
        )

    print(
        f"Processed test chunk {chunk_number}: "
        f"{len(chunk):,} records"
    )

# Combine results after all chunks finish
y_test_70 = np.concatenate(true_labels_70)

assert len(y_test_70) == 421_559

results_70 = []

for name in model_names:
    y_pred_test = np.concatenate(
        predictions_70[name]
    )

    assert len(y_pred_test) == len(y_test_70)

    tn, fp, fn, tp = confusion_matrix(
        y_test_70,
        y_pred_test,
        labels=[0, 1]
    ).ravel()

    results_70.append({
        "Model": name,
        "Features": len(final_models[name]["features"]),
        "Accuracy": accuracy_score(
            y_test_70, y_pred_test
        ),
        "Attack_precision": precision_score(
            y_test_70, y_pred_test,
            zero_division=0
        ),
        "Attack_recall": recall_score(
            y_test_70, y_pred_test,
            zero_division=0
        ),
        "Macro_F1": f1_score(
            y_test_70, y_pred_test,
            average="macro",
            zero_division=0
        ),
        "FPR": fp / (fp + tn),
        "Missed_attacks": int(fn),
        "False_alarms": int(fp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "Prediction_seconds": round(
            prediction_times_70[name], 3
        )
    })

summary_70 = pd.DataFrame(results_70)

print("\nFINAL TEST RESULTS")
print("=" * 65)

display_columns = [
    "Model",
    "Features",
    "Accuracy",
    "Attack_precision",
    "Attack_recall",
    "Macro_F1",
    "FPR",
    "Missed_attacks",
    "False_alarms",
    "Prediction_seconds"
]

print(
    summary_70[display_columns]
    .to_string(index=False)
)

print("\nTest records:", len(y_test_70))
print("All three models evaluated: True")
print("\nExperiment 70 completed.")


In [ ]:
import pandas as pd
import numpy as np

print("CICIDS2017 — Experiment 71")
print("FINAL TEST — Attack-Category Recall")
print("=" * 65)

# Read labels only, preserving the exact test-file row order
test_categories_71 = []

for chunk in pd.read_csv(
    test_path,
    usecols=lambda c: c.strip() == "Label",
    chunksize=50_000
):
    chunk.columns = chunk.columns.str.strip()
    test_categories_71.extend(
        chunk["Label"].astype(str).str.strip().tolist()
    )

test_categories_71 = np.asarray(test_categories_71)

# Reuse predictions from Experiment 70
pred_71 = {
    name: np.concatenate(predictions_70[name])
    for name in model_names
}

assert len(test_categories_71) == 421_559

# Check that the reconstructed binary labels match
assert np.array_equal(
    (test_categories_71 != "BENIGN").astype(np.int8),
    y_test_70
)

attack_mask_71 = test_categories_71 != "BENIGN"

category_71 = pd.DataFrame({
    "Attack_category": test_categories_71[attack_mask_71]
})

for name in model_names:
    category_71[name] = pred_71[name][attack_mask_71]

summary_71 = (
    category_71.groupby("Attack_category")
    .agg(
        Total=("All-78", "size"),
        All78_detected=("All-78", "sum"),
        MI30_detected=("MI-30", "sum"),
        MI50_detected=("MI-50", "sum")
    )
    .reset_index()
)

for short in ["All78", "MI30", "MI50"]:
    summary_71[f"{short}_missed"] = (
        summary_71["Total"]
        - summary_71[f"{short}_detected"]
    )

    summary_71[f"{short}_recall"] = (
        summary_71[f"{short}_detected"]
        / summary_71["Total"]
    )

summary_71 = summary_71.sort_values("All78_recall")

columns_71 = [
    "Attack_category", "Total",
    "All78_missed", "MI30_missed", "MI50_missed",
    "All78_recall", "MI30_recall", "MI50_recall"
]

print("\nPer-category test results:")
print(summary_71[columns_71].to_string(index=False))

print("\nTotal missed attacks:")
for short in ["All78", "MI30", "MI50"]:
    print(short, int(summary_71[f"{short}_missed"].sum()))

assert int(summary_71["All78_missed"].sum()) == 355
assert int(summary_71["MI30_missed"].sum()) == 602
assert int(summary_71["MI50_missed"].sum()) == 668

print("\nExperiment 71 completed.")

In [ ]:
import json
from pathlib import Path

print("CICIDS2017 — Experiment 72")
print("Save Final Held-Out Test Evidence")
print("=" * 60)

output_72 = data_folder.parent / "experiment72"
output_72.mkdir(parents=True, exist_ok=True)

performance_file = output_72 / "final_test_performance.csv"
categories_file = output_72 / "final_test_attack_categories.csv"
record_file = output_72 / "final_test_research_record.json"

# Save the previously calculated results
summary_70.to_csv(performance_file, index=False)
summary_71.to_csv(categories_file, index=False)

record_72 = {
    "dataset": "CICIDS2017",
    "evaluation": "Frozen duplicate-aware held-out test",
    "test_records": 421559,
    "models": ["All-78", "MI-30", "MI-50"],
    "model_selection": (
        "Models and feature sets were selected before "
        "the final held-out test evaluation."
    ),
    "limitations": [
        "The test partition comes from the same CICIDS2017 collection.",
        "Duplicate-aware splitting does not guarantee capture-session independence.",
        "Rare attack categories have small test sample sizes.",
        "The test results must not be used for further model tuning."
    ]
}

with open(record_file, "w") as f:
    json.dump(record_72, f, indent=2)

print("Performance CSV saved:", performance_file.exists())
print("Attack-category CSV saved:", categories_file.exists())
print("Research record saved:", record_file.exists())

assert performance_file.exists()
assert categories_file.exists()
assert record_file.exists()

print("\nExperiment 72 completed.")
